# Projet 7 – Étape 2 : Extraction de features avec ResNet

**Objectif :** Transformer chaque image en un vecteur de 2048 nombres  
qui résument ses caractéristiques visuelles.

**Principe :**
- On charge ResNet50 pré-entraîné sur ImageNet
- On coupe sa dernière couche (celle qui classifie)
- On passe toutes les images dedans → chacune ressort comme un vecteur de 2048 valeurs
- On sauvegarde ces vecteurs pour l'étape 3 (clustering)

**Important :** les données labelisées et sans label sont traitées séparément  
et ne seront jamais mélangées.

## 0. Imports et configuration

In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torchvision.models as models
import torchvision.transforms as transforms

# Chemins
ROOT        = Path(".")
DIR_CANCER  = ROOT / "avec_labels" / "cancer"
DIR_NORMAL  = ROOT / "avec_labels" / "normal"
DIR_UNLABEL = ROOT / "sans_label"
DIR_OUTPUT  = ROOT / "output"   # dossier pour sauvegarder les features
DIR_OUTPUT.mkdir(exist_ok=True)

# CPU only
DEVICE = torch.device("cpu")
print(f"Device : {DEVICE}")

Device : cpu


## 1. Préparation des images

ResNet attend des images :
- Redimensionnées à **224×224**
- Normalisées avec les stats ImageNet (mean et std par canal RGB)

Ces stats viennent du dataset sur lequel ResNet a été entraîné.  
On les réutilise telles quelles — c'est la convention.

In [3]:
# Pipeline de transformation appliqué à chaque image
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),                        # PIL → tenseur [0,1]
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],               # stats ImageNet canal R, G, B
        std=[0.229, 0.224, 0.225]
    )
])

print("Pipeline de transformation prêt.")

Pipeline de transformation prêt.


## 2. Chargement de ResNet50

On charge ResNet50 pré-entraîné, puis on **supprime la dernière couche**  
(le classifieur final) pour ne garder que l'extracteur de features.

In [4]:
# Chargement de ResNet50 pré-entraîné
resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

# On coupe la dernière couche (fc = fully connected = le classifieur)
# nn.Identity() = "ne fais rien", laisse passer le vecteur de 2048 tel quel
import torch.nn as nn
resnet.fc = nn.Identity()

# Mode évaluation : on n'entraîne pas, on extrait seulement
resnet.eval()
resnet = resnet.to(DEVICE)

print("ResNet50 chargé. Dernière couche remplacée par Identity.")
print("Taille de sortie attendue : 2048")

ResNet50 chargé. Dernière couche remplacée par Identity.
Taille de sortie attendue : 2048


In [10]:
# Chargement de ResNet50 pré-entraîné
from torchsummary import summary
resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
summary(resnet, (3, 224, 224))


----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 64, 112, 112]           9,408
       BatchNorm2d-2         [-1, 64, 112, 112]             128
              ReLU-3         [-1, 64, 112, 112]               0
         MaxPool2d-4           [-1, 64, 56, 56]               0
            Conv2d-5           [-1, 64, 56, 56]           4,096
       BatchNorm2d-6           [-1, 64, 56, 56]             128
              ReLU-7           [-1, 64, 56, 56]               0
            Conv2d-8           [-1, 64, 56, 56]          36,864
       BatchNorm2d-9           [-1, 64, 56, 56]             128
             ReLU-10           [-1, 64, 56, 56]               0
           Conv2d-11          [-1, 256, 56, 56]          16,384
      BatchNorm2d-12          [-1, 256, 56, 56]             512
           Conv2d-13          [-1, 256, 56, 56]          16,384
      BatchNorm2d-14          [-1, 256,

## 3. Fonction d'extraction

On définit une fonction qui prend une liste d'images  
et retourne un tableau numpy de shape `(n_images, 2048)`.

In [5]:
def extraire_features(liste_chemins, modele, transform, device, batch_size=32):
    """
    Extrait les features de toutes les images avec ResNet.
    Traite les images par batch pour ne pas saturer la mémoire.
    
    Retourne :
        features : numpy array de shape (n_images, 2048)
        noms     : liste des noms de fichiers (pour traçabilité)
    """
    features = []
    noms = []
    
    n_total = len(liste_chemins)
    
    with torch.no_grad():   # pas de calcul de gradient, on n'entraîne pas
        for i in range(0, n_total, batch_size):
            batch_chemins = liste_chemins[i:i+batch_size]
            batch_tenseurs = []
            
            for chemin in batch_chemins:
                img = Image.open(chemin).convert("RGB")  # force RGB au cas où
                tenseur = transform(img)
                batch_tenseurs.append(tenseur)
                noms.append(chemin.name)
            
            # Empile les tenseurs en un batch : shape (batch_size, 3, 224, 224)
            batch = torch.stack(batch_tenseurs).to(device)
            
            # Passe dans ResNet → shape (batch_size, 2048)
            output = modele(batch)
            features.append(output.cpu().numpy())
            
            # Progression
            print(f"  {min(i+batch_size, n_total)}/{n_total} images traitées", end="\r")
    
    print()  # saut de ligne final
    return np.vstack(features), noms

print("Fonction d'extraction définie.")

Fonction d'extraction définie.


## 4. Extraction sur les données labelisées

On traite cancer et normal séparément, on garde les labels.

In [6]:
images_cancer  = sorted(DIR_CANCER.glob("*.jpg"))
images_normal  = sorted(DIR_NORMAL.glob("*.jpg"))

print("Extraction des images cancer...")
features_cancer, noms_cancer = extraire_features(images_cancer, resnet, transform, DEVICE)

print("Extraction des images normal...")
features_normal, noms_normal = extraire_features(images_normal, resnet, transform, DEVICE)

# On assemble et on crée les labels
features_label = np.vstack([features_cancer, features_normal])
labels         = np.array([1]*len(images_cancer) + [0]*len(images_normal))  # 1=cancer, 0=normal
noms_label     = noms_cancer + noms_normal

print(f"\nShape features labelisées : {features_label.shape}")
print(f"Distribution labels : {dict(zip(*np.unique(labels, return_counts=True)))}")

Extraction des images cancer...
  50/50 images traitées
Extraction des images normal...
  50/50 images traitées

Shape features labelisées : (100, 2048)
Distribution labels : {np.int64(0): np.int64(50), np.int64(1): np.int64(50)}


## 5. Extraction sur les données sans label

In [7]:
images_unlabel = sorted(DIR_UNLABEL.glob("*.jpg"))

print("Extraction des images sans label (ça peut prendre quelques minutes)...")
features_unlabel, noms_unlabel = extraire_features(images_unlabel, resnet, transform, DEVICE)

print(f"\nShape features sans label : {features_unlabel.shape}")

Extraction des images sans label (ça peut prendre quelques minutes)...
  1406/1406 images traitées

Shape features sans label : (1406, 2048)


## 6. Vérification rapide

On s'assure que les vecteurs ont bien la bonne forme  
et qu'ils ne sont pas tous identiques (ce qui indiquerait un bug).

In [8]:
print("=== Vérification ===")
print(f"Features labelisées  : {features_label.shape}   → attendu : (100, 2048)")
print(f"Features sans label  : {features_unlabel.shape} → attendu : (1406, 2048)")
print()
print(f"Valeurs min/max (labelisé) : {features_label.min():.3f} / {features_label.max():.3f}")
print(f"Valeurs min/max (unlabel)  : {features_unlabel.min():.3f} / {features_unlabel.max():.3f}")
print()
# Vérifie que les vecteurs sont différents entre eux (variance > 0)
print(f"Variance moyenne des features labelisées : {features_label.var(axis=0).mean():.4f}")

=== Vérification ===
Features labelisées  : (100, 2048)   → attendu : (100, 2048)
Features sans label  : (1406, 2048) → attendu : (1406, 2048)

Valeurs min/max (labelisé) : 0.000 / 6.542
Valeurs min/max (unlabel)  : 0.000 / 7.743

Variance moyenne des features labelisées : 0.0348


## 7. Sauvegarde

On sauvegarde les features dans `output/` pour les réutiliser à l'étape 3  
sans avoir à relancer ResNet.

In [ ]:
# Sauvegarde numpy
np.save(DIR_OUTPUT / "features_label.npy",   features_label)
np.save(DIR_OUTPUT / "features_unlabel.npy", features_unlabel)
np.save(DIR_OUTPUT / "labels.npy",           labels)

# Sauvegarde des noms de fichiers (pour traçabilité)
pd.DataFrame({"fichier": noms_label,   "label": labels}).to_csv(DIR_OUTPUT / "meta_label.csv",   index=False)
pd.DataFrame({"fichier": noms_unlabel}).to_csv(DIR_OUTPUT / "meta_unlabel.csv", index=False)

print("Fichiers sauvegardés dans le dossier output/ :")
for f in sorted(DIR_OUTPUT.iterdir()):
    print(f"  {f.name}  ({f.stat().st_size/1024:.0f} Ko)")

Fichiers sauvegardés dans zesaispa/ :
  features_label.npy  (800 Ko)
  features_unlabel.npy  (11248 Ko)
  labels.npy  (1 Ko)
  meta_label.csv  (4 Ko)
  meta_unlabel.csv  (58 Ko)
